In [ ]:
import os
import json
import traceback

JSON_FOLDER_PATH = "./" 

def extract_exact_immarkus_data(folder_path):
    total_N_total = 0
    total_Insertion = 0
    total_Deletion = 0
    total_Substitution = 0
    
    pathology_stats = {}
    
    try:
        all_files = os.listdir(folder_path)
    except Exception as e:
        print(f"Failed")
        return
    
    json_files = [f for f in all_files if f.lower().endswith('.json')]
    processed_files_count = 0

    for filename in json_files:
        file_path = os.path.join(folder_path, filename)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if not isinstance(data, list):
                data_list = [data]
            else:
                data_list = data
                
            file_has_valid_data = False
            line_count_in_file = 0
            
            for annotation in data_list:
                if not isinstance(annotation, dict):
                    continue
                
                body_list = annotation.get('body', [])
                for body_item in body_list:
                    props = body_item.get('properties', {})
                    if not props:
                        continue
                    
                    n_total = props.get('N_total_line')
                    if n_total is not None:
                        total_N_total += int(n_total)
                        total_Insertion += int(props.get('Insertion_Count', 0))
                        total_Deletion += int(props.get('Deletion_Count', 0))
                        total_Substitution += int(props.get('Substitution_Count', 0))
                        file_has_valid_data = True
                        line_count_in_file += 1
                        
                        # Analyze the distribution of error.
                        pathology_data = props.get('Error_Pathology')
                        if pathology_data:
                            if isinstance(pathology_data, list):
                                pathology_list = pathology_data
                            else:
                                pathology_list = [p.strip() for p in str(pathology_data).split(',') if p.strip()]
                            for path_item in pathology_list:
                                if path_item and path_item.lower() != 'none':
                                    pathology_stats[path_item] = pathology_stats.get(path_item, 0) + 1
            
            if file_has_valid_data:
                processed_files_count += 1
                print(f"Successfully parsed file [{processed_files_count}]: {filename} ({line_count_in_file} annotated lines read successfully)")
            else:
                print(f"File {filename} loaded successfully, but valid metrics such as N_total_line were not found in its internal structure.")

        except Exception as e:
            print(f"An exception occurred while parsing file {filename}. Error stack trace:")
            traceback.print_exc()

    print("-"*50 + f"\n Scan finished! Total annotated files counted into metrics: {processed_files_count}")
    
    if total_N_total == 0:
        print("\nError Prompt: No valid data detected. Please verify file contents or check if field names match exactly.")
        return

    ar_star = (total_N_total - total_Insertion - total_Deletion - total_Substitution) / total_N_total
    cr_star = (total_N_total - total_Deletion - total_Substitution) / total_N_total

    print(f"[Overall Basic Quantitative Scale]")
    print(f"   - Total Real Characters (N_total*): {total_N_total} chars")
    print(f"[Algorithm Model Recognition Errors]")
    print(f"   - Total Insertion Errors (N_Ie*):     {total_Insertion} chars")
    print(f"   - Total Deletion Errors (N_De*):     {total_Deletion} chars")
    print(f"   - Total Substitution Errors (N_Se*):     {total_Substitution} chars")
    print(f"[Improved Core Evaluation Metrics]")
    print(f"   -  Improved Accuracy Rate (AR*): {ar_star * 100:.2f}%")
    print(f"   -  Improved Correct Rate (CR*):  {cr_star * 100:.2f}%")
    print(f"[Macro Distribution of Error Root Causes / Pathological Phenomena (Count of Disturbed Lines)]")
    if pathology_stats:
        for err_type, count in pathology_stats.items():
            print(f"   - {err_type}: {count} lines affected")
    else:
        print("   - Perfect performance! All bounding boxes are fully correct (None).")
    print("="*66)

if __name__ == "__main__":
    extract_exact_immarkus_data(JSON_FOLDER_PATH)